# LeetCode 2189 — Number of Ways to Build House of Cards bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-to-build-house-of-cards/ · Độ khó: **Medium** (🔒 Premium) · Chủ đề: Math, Dynamic Programming

> **Lưu ý Premium:** đây là bài Premium; đề bài trong notebook được viết theo bản mô tả công khai của doocs/leetcode. Nếu bạn có tài khoản Premium, hãy **đối chiếu lại với đề gốc** (tên hàm, ràng buộc, ví dụ, hình minh họa).

Skeleton LeetCode:

```python
class Solution:
    def houseOfCards(self, n: int) -> int:
        ...
```

Mô hình hóa: một hàng có `k` tam giác dùng `2k` lá dựng + `k − 1` lá nằm ngang = **`3k − 1`** lá. Hàng phía trên phải có **ít tam giác hơn hẳn** hàng dưới. ⇒ Đếm số **dãy giảm nghiêm ngặt** `k₁ > k₂ > … > k_r ≥ 1` với `Σ (3kᵢ − 1) = n`. Notebook đi từ **quay lui liệt kê mọi nhà** → **DFS ghi nhớ `(còn lại, tối đa)`** → **knapsack 0/1 "mỗi cỡ hàng dùng tối đa một lần"** → **đổi biến sang phân hoạch thành phần tử phân biệt** → **DP đếm phân hoạch với đúng `r` phần**.

## 1. Mục tiêu học

1. Đọc hiểu một mô tả hình học và rút ra **công thức số lá** của một hàng: `3k − 1`.
2. Suy ra ràng buộc giữa các hàng: tam giác hàng trên đặt trên lá ngang hàng dưới ⇒ `k_trên ≤ k_dưới − 1`.
3. Đếm **tập** các cỡ hàng (vì dãy giảm nghiêm ngặt ↔ tập) ⇒ knapsack 0/1 đếm số cách.
4. Biến đổi `Σ(3k − 1) = n` thành bài **phân hoạch thành `r` phần phân biệt** của `(n + r) / 3`.
5. Kiểm thử bằng quay lui + so khớp các ví dụ của đề (`16 → 2`, `2 → 1`, `4 → 0`).

## 2. Đề bài & ví dụ

Có `n` lá bài. Một **nhà bài** thỏa:
- gồm một hoặc nhiều **hàng** tam giác và lá nằm ngang;
- tam giác = hai lá dựng tựa vào nhau;
- giữa **hai tam giác kề nhau** trong một hàng phải có **một lá nằm ngang**;
- mọi tam giác ở hàng cao hơn hàng đầu tiên phải đặt **trên một lá nằm ngang** của hàng dưới;
- mỗi tam giác đặt ở vị trí **trái nhất** còn trống trong hàng.

Đếm số nhà bài **khác nhau** dùng **đúng hết** `n` lá (hai nhà khác nhau nếu có một hàng có số lá khác nhau).

**Ví dụ 1:** `n = 16` → `2`.

**Ví dụ 2:** `n = 2` → `1`.

**Ví dụ 3:** `n = 4` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 500`.

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [(16, 2), (2, 1), (4, 0)]

def row_cards(k):
    return 2 * k + (k - 1)          # 2 lá mỗi tam giác + (k - 1) lá ngang ở giữa

print("số lá theo số tam giác k:", [(k, row_cards(k)) for k in range(1, 8)])

số lá theo số tam giác k: [(1, 2), (2, 5), (3, 8), (4, 11), (5, 14), (6, 17), (7, 20)]


## 3. Kiến thức nền

### 3.1 Vẽ một hàng để đếm lá

In [2]:
def draw_row(k):
    return "_".join(["/\\"] * k)          # '/\' là tam giác, '_' là lá nằm ngang

for k in range(1, 5):
    print(f"  k={k}: {draw_row(k):<12} -> {row_cards(k)} lá")

  k=1: /\           -> 2 lá
  k=2: /\_/\        -> 5 lá
  k=3: /\_/\_/\     -> 8 lá
  k=4: /\_/\_/\_/\  -> 11 lá


### 3.2 Vì sao hàng trên phải ít tam giác hơn **hẳn**

Hàng dưới có `k` tam giác thì có `k − 1` lá ngang (giữa các tam giác) — đó là các "chỗ đặt" cho tam giác của hàng trên. Hàng trên có `k'` tam giác cần `k'` chỗ đặt ⇒ `k' ≤ k − 1`. Ngược lại, với mọi `k' ≤ k − 1` đều xếp được (đặt từ trái sang — luật "vị trí trái nhất" khiến cách đặt là **duy nhất** với mỗi `k'`).

In [3]:
def draw_house(rows):                 # rows: từ dưới lên, giảm nghiêm ngặt
    width = 3 * rows[0]
    lines = []
    for level, k in enumerate(rows):
        lines.append(" " * (level * 3 // 2 + level) + draw_row(k))
    return "\n".join(reversed(lines))

print(draw_house([3, 2]), "\n   (3 tam giác dưới: 8 lá; 2 tam giác trên: 5 lá; tổng 13)")

  /\_/\
/\_/\_/\ 
   (3 tam giác dưới: 8 lá; 2 tam giác trên: 5 lá; tổng 13)


### 3.3 Dãy giảm nghiêm ngặt ↔ tập hợp

Mỗi dãy `k₁ > k₂ > … > k_r` tương ứng đúng một **tập** `{k₁, …, k_r}` (sắp giảm lại là ra dãy). ⇒ Đếm nhà bài = đếm **tập con** của `{1, 2, 3, …}` có `Σ (3k − 1) = n`.

### 3.4 Knapsack 0/1 đếm số cách

`ways[s]` = số tập con của các vật đã xét có tổng chi phí `s`. Thêm vật chi phí `c`: duyệt `s` **giảm dần** `ways[s] += ways[s − c]` (mỗi vật dùng tối đa một lần).

In [4]:
ways = [1] + [0] * 10
for c in [2, 3, 5]:                    # đếm tập con có tổng s
    for s in range(10, c - 1, -1):
        ways[s] += ways[s - c]
print("số tập con của {2,3,5} theo tổng:", list(enumerate(ways)))

số tập con của {2,3,5} theo tổng: [(0, 1), (1, 0), (2, 1), (3, 1), (4, 0), (5, 2), (6, 0), (7, 1), (8, 1), (9, 0), (10, 1)]


## 4. Ý tưởng nền tảng / trực giác

Số lá của từng cỡ hàng: `k = 1 → 2`, `2 → 5`, `3 → 8`, `4 → 11`, `5 → 14`, …

Tính tay:
- `n = 2`: chỉ `{1}` (2 lá) ⇒ 1 ✓.
- `n = 4`: cần tập có tổng 4 từ `{2, 5, 8, …}` — `2 + 2` không được (cỡ phải khác nhau) ⇒ 0 ✓.
- `n = 16`: `{3, 2, 1}` ⇒ `8 + 5 + 2 = 15` ✗; `{5, 1}` ⇒ `14 + 2 = 16` ✓; `{4, 2}` ⇒ `11 + 5 = 16` ✓; `{3, 2}` = 13 ✗… ⇒ 2 ✓.

Nhận xét: một nhà `r` hàng dùng `3 Σk − r` lá ⇒ `n + r` phải chia hết cho 3.

In [5]:
def list_houses(n):
    out = []
    def go(remain, max_k, cur):
        if remain == 0:
            out.append(tuple(cur)); return
        for k in range(min(max_k, (remain + 1) // 3), 0, -1):
            if row_cards(k) <= remain:
                go(remain - row_cards(k), k - 1, cur + [k])
    go(n, n, [])
    return out

for n_ in [2, 4, 16, 30]:
    hs = list_houses(n_)
    print(f"  n={n_}: {len(hs)} nhà -> {hs}")

  n=2: 1 nhà -> [(1,)]
  n=4: 0 nhà -> []
  n=16: 2 nhà -> [(5, 1), (4, 2)]
  n=30: 5 nhà -> [(8, 2, 1), (7, 3, 1), (6, 4, 1), (6, 3, 2), (5, 4, 2)]


## 5. Cách 1 — Brute Force / baseline: quay lui liệt kê mọi nhà

Chọn hàng dưới cùng `k₁`, rồi hàng trên `k₂ < k₁`, … cho tới khi dùng hết lá. Đếm số lần dùng đúng hết.

In [6]:
def house_brute(n):
    return len(list_houses(n))

for n_, exp in examples:
    print(house_brute(n_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp:** bằng số nhà cộng các nhánh cụt — tăng nhanh theo `n` (số phân hoạch thành phần phân biệt tăng như `e^{c√n}`). Với `n = 500` vẫn chạy được (các cỡ `k ≤ 167`), nhưng không có ghi nhớ thì nhiều nhánh bị duyệt lặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái quay lui chỉ phụ thuộc** `(số lá còn lại, cỡ hàng tối đa được phép)` ⇒ ghi nhớ.

`f(remain, m)` = số cách dùng hết `remain` lá với các hàng có cỡ **≤ m**, giảm nghiêm ngặt:
- `f(0, m) = 1`;
- **nếu** `remain > 0` và `m = 0` **thì** 0;
- **ngược lại** `f(remain, m) = f(remain, m − 1) + [3m − 1 ≤ remain] · f(remain − (3m − 1), m − 1)` (không dùng / dùng cỡ `m`).

Đáp án `f(n, n)`. Đây chính là **knapsack 0/1 đếm số cách** viết dưới dạng đệ quy.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS ghi nhớ `(remain, m)`

In [7]:
def house_memo(n):
    @lru_cache(maxsize=None)
    def f(remain, m):
        if remain == 0:
            return 1
        if m == 0:
            return 0
        res = f(remain, m - 1)
        c = 3 * m - 1
        if c <= remain:
            res += f(remain - c, m - 1)
        return res
    kmax = (n + 1) // 3
    return f(n, kmax)

for n_, exp in examples:
    print(house_memo(n_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n · kmax) = O(n²/3)` trạng thái. **Độ sâu đệ quy** tới `kmax ≈ 167` — an toàn.

### 7.2 Cách 3 — Knapsack 0/1 đếm số cách (lặp)

In [8]:
def house_knapsack(n):
    ways = [1] + [0] * n
    k = 1
    while 3 * k - 1 <= n:
        c = 3 * k - 1
        for s in range(n, c - 1, -1):     # giảm dần: mỗi cỡ dùng tối đa 1 lần
            ways[s] += ways[s - c]
        k += 1
    return ways[n]

for n_, exp in examples:
    print(house_knapsack(n_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n · kmax)` thời gian, `O(n)` bộ nhớ. Một lần chạy cho **mọi** `s ≤ n` (bảng `ways`).

### 7.3 Cách 4 — Đổi biến: phân hoạch `(n + r)/3` thành `r` phần phân biệt

`Σ(3kᵢ − 1) = n` với `r` hàng ⇔ `Σ kᵢ = (n + r) / 3`. ⇒ Đáp án `= Σ_r P_dist((n + r)/3, r)` trên các `r` có `(n + r) % 3 == 0`, với `P_dist(S, r)` = số cách viết `S` thành tổng `r` số nguyên dương **phân biệt**.

`P_dist(S, r)` tính bằng DP: `D[r][S] = D[r][S − r] + D[r − 1][S − r]` (trừ 1 khỏi mọi phần; phần nhỏ nhất bằng 1 thì biến mất).

In [9]:
def distinct_partitions_table(Smax, rmax):
    D = [[0] * (Smax + 1) for _ in range(rmax + 1)]
    D[0][0] = 1
    for r in range(1, rmax + 1):
        for S in range(r, Smax + 1):
            D[r][S] = D[r][S - r] + D[r - 1][S - r]
    return D

def house_partitions(n):
    rmax = 1
    while 3 * rmax * (rmax + 1) // 2 - rmax <= n:     # nhà nhỏ nhất r hàng: cỡ 1..r
        rmax += 1
    Smax = (n + rmax) // 3
    D = distinct_partitions_table(Smax, rmax)
    total = 0
    for r in range(1, rmax + 1):
        if (n + r) % 3 == 0:
            total += D[r][(n + r) // 3]
    return total

for n_, exp in examples:
    print(house_partitions(n_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Vì sao truy hồi `D[r][S]` đúng?** Lấy `r` phần phân biệt `a₁ > … > a_r ≥ 1`, trừ 1 khỏi mỗi phần: nếu `a_r ≥ 2` được `r` phần phân biệt tổng `S − r`; nếu `a_r = 1` được `r − 1` phần phân biệt (phần cuối thành 0, bỏ đi) tổng `S − r`. Song ánh ⇒ cộng hai trường hợp. **Độ phức tạp:** `O(rmax · S)` với `rmax ≈ √(2n/3) ≈ 18` ⇒ rất nhỏ.

### 7.4 Cách 5 — Sinh hàm (kiểm tra bằng nhân đa thức)

Đáp án là hệ số `xⁿ` của `Π_{k ≥ 1} (1 + x^{3k−1})`. Nhân lần lượt các đa thức (cắt bậc `> n`) — thực chất trùng cách 3, nhưng cách nhìn này giúp viết công thức cho các biến thể (ví dụ cho phép hàng trùng cỡ ⇒ `Π 1/(1 − x^{3k−1})`).

In [10]:
def house_generating(n):
    poly = [1] + [0] * n
    for k in range(1, (n + 1) // 3 + 1):
        c = 3 * k - 1
        new = poly[:]
        for d in range(n - c + 1):
            if poly[d]:
                new[d + c] += poly[d]
        poly = new
    return poly[n]

for n_, exp in examples:
    print(house_generating(n_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


## 8. TRACE từng bước

### 8.1 Trace knapsack với `n = 16` — bảng `ways` sau mỗi cỡ hàng

In [11]:
def trace_knapsack(n):
    ways = [1] + [0] * n
    k = 1
    print("   s    :", " ".join(f"{s:>2}" for s in range(n + 1)))
    while 3 * k - 1 <= n:
        c = 3 * k - 1
        for s in range(n, c - 1, -1):
            ways[s] += ways[s - c]
        print(f"  k={k} ({c:>2}):", " ".join(f"{w:>2}" for w in ways))
        k += 1
    return ways[n]

print("=>", trace_knapsack(16))

   s    :  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16
  k=1 ( 2):  1  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  k=2 ( 5):  1  0  1  0  0  1  0  1  0  0  0  0  0  0  0  0  0
  k=3 ( 8):  1  0  1  0  0  1  0  1  1  0  1  0  0  1  0  1  0
  k=4 (11):  1  0  1  0  0  1  0  1  1  0  1  1  0  2  0  1  1
  k=5 (14):  1  0  1  0  0  1  0  1  1  0  1  1  0  2  1  1  2
=> 2


### 8.2 Trace đổi biến với `n = 30` (ví dụ tự dựng — đã đối chiếu quay lui, đối chiếu lại đề gốc)

In [12]:
def trace_partitions(n):
    rmax = 1
    while 3 * rmax * (rmax + 1) // 2 - rmax <= n: rmax += 1
    D = distinct_partitions_table((n + rmax) // 3, rmax)
    total = 0
    for r in range(1, rmax + 1):
        if (n + r) % 3:
            print(f"    r={r}: (n + r) = {n + r} không chia hết cho 3 -> bỏ"); continue
        S = (n + r) // 3
        total += D[r][S]
        print(f"    r={r}: Σk = {S}, số cách chia {S} thành {r} phần phân biệt = {D[r][S]}")
    return total

r = trace_partitions(30)
print("  =>", r, "| quay lui:", house_brute(30), list_houses(30))

    r=1: (n + r) = 31 không chia hết cho 3 -> bỏ
    r=2: (n + r) = 32 không chia hết cho 3 -> bỏ
    r=3: Σk = 11, số cách chia 11 thành 3 phần phân biệt = 5
    r=4: (n + r) = 34 không chia hết cho 3 -> bỏ
    r=5: (n + r) = 35 không chia hết cho 3 -> bỏ
  => 5 | quay lui: 5 [(8, 2, 1), (7, 3, 1), (6, 4, 1), (6, 3, 2), (5, 4, 2)]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def houseOfCards(self, n: int) -> int:
        ways = [1] + [0] * n                 # ways[s]: số tập cỡ hàng có tổng số lá = s
        k = 1
        while 3 * k - 1 <= n:
            c = 3 * k - 1                    # hàng k tam giác: 2k lá dựng + (k - 1) lá ngang
            for s in range(n, c - 1, -1):    # 0/1: mỗi cỡ hàng dùng tối đa một lần
                ways[s] += ways[s - c]
            k += 1
        return ways[n]

for n_, exp in examples:
    assert Solution().houseOfCards(n_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Số lá mỗi hàng.** Hàng `k` tam giác cần `2k` lá dựng và `k − 1` lá ngang (một giữa mỗi cặp tam giác kề) ⇒ `3k − 1`.
2. **Ràng buộc giữa hàng.** Tam giác hàng trên cần lá ngang của hàng dưới làm đế; hàng dưới có `k − 1` lá ngang ⇒ hàng trên `≤ k − 1` tam giác. Luật "trái nhất" làm cho vị trí cụ thể được xác định, nên một nhà được xác định hoàn toàn bởi **dãy số tam giác** `k₁ > k₂ > …` (và hai nhà khác nhau ⇔ dãy khác nhau, vì số lá mỗi hàng `3k − 1` xác định `k`).
3. **Dãy giảm nghiêm ngặt ↔ tập.** ⇒ Đếm tập `K ⊂ ℕ⁺` với `Σ_{k ∈ K}(3k − 1) = n`.
4. **Knapsack 0/1:** bất biến sau khi xét các cỡ `1..k`: `ways[s]` = số tập con của `{1..k}` có tổng chi phí `s` (duyệt `s` giảm dần để vật `k` không bị dùng hai lần).
5. Các cỡ `k` có `3k − 1 > n` không thể dùng ⇒ dừng vòng lặp là đủ.

## 11. Phân tích Time / Space Complexity

Gọi `K = ⌊(n + 1)/3⌋` (cỡ hàng lớn nhất khả dĩ), `R ≈ √(2n/3)` (số hàng tối đa).

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | ≥ số nhà (mũ theo `√n`) | `O(R)` |
| 2. DFS ghi nhớ | `O(n K)` | `O(n K)` |
| 3. Knapsack 0/1 | `O(n K)` | `O(n)` |
| 4. Phân hoạch phân biệt | `O(R · n/3)` | `O(R · n/3)` |
| 5. Sinh hàm | `O(n K)` | `O(n)` |

In [14]:
t0 = time.perf_counter(); rb = house_brute(200); print(f"brute      : n=200 -> {rb}  ({(time.perf_counter() - t0) * 1000:.1f} ms)  (n=500 có ~2.5·10⁷ nhà -> quá chậm để liệt kê)")
for name, fn in [("memo", house_memo), ("knapsack", house_knapsack),
                 ("partitions", house_partitions), ("generating", house_generating), ("Solution", Solution().houseOfCards)]:
    t0 = time.perf_counter(); r = fn(500); dt = time.perf_counter() - t0
    print(f"{name:<11}: n=500 -> {r}  ({dt * 1000:.1f} ms)")

brute      : n=200 -> 9090  (72.1 ms)  (n=500 có ~2.5·10⁷ nhà -> quá chậm để liệt kê)
memo       : n=500 -> 25008984  (8.6 ms)
knapsack   : n=500 -> 25008984  (1.7 ms)
partitions : n=500 -> 25008984  (0.1 ms)
generating : n=500 -> 25008984  (1.8 ms)
Solution   : n=500 -> 25008984  (1.6 ms)


## 12. Bộ test

In [15]:
ALL = {
    "memo": house_memo,
    "knapsack": house_knapsack,
    "partitions": house_partitions,
    "generating": house_generating,
    "Solution": lambda n: Solution().houseOfCards(n),
}

def check(n, exp=None, brute=True):
    ref = house_brute(n) if brute else house_memo(n)
    for name, fn in ALL.items():
        got = fn(n)
        assert got == ref, (name, n, got, ref)
    if exp is not None:
        assert ref == exp, (n, ref, exp)
    return ref

for n_, exp in examples:
    check(n_, exp)
print("Ví dụ đề: OK")

check(1, 0); check(3, 0); check(5, 1); check(7, 1)        # 7 = 5 + 2
check(13, 2)                                                # 8 + 5 và 11 + 2
check(15, 1)                                                # chỉ 8 + 5 + 2
print("Edge case nhỏ (kỳ vọng tự dựng, đã đối chiếu quay lui): OK")

for n_ in range(1, 201):
    check(n_)
print("Mọi n ∈ [1, 200] — 5 cách khớp quay lui: OK")
for n_ in range(201, 501):
    check(n_, brute=False)
print("Mọi n ∈ [201, 500] — đối chiếu chéo 5 cách (quay lui quá chậm ở đây): OK")
print("Một số giá trị:", {n_: Solution().houseOfCards(n_) for n_ in [10, 50, 100, 200, 300, 400, 500]})

Ví dụ đề: OK
Edge case nhỏ (kỳ vọng tự dựng, đã đối chiếu quay lui): OK


Mọi n ∈ [1, 200] — 5 cách khớp quay lui: OK


Mọi n ∈ [201, 500] — đối chiếu chéo 5 cách (quay lui quá chậm ở đây): OK
Một số giá trị: {10: 1, 50: 16, 100: 209, 200: 9090, 300: 184941, 400: 2480683, 500: 25008984}


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số lá một hàng là `3k` (quên trừ 1) hoặc `2k`

In [16]:
def bug_cost_3k(n):
    ways = [1] + [0] * n; k = 1
    while 3 * k <= n:
        c = 3 * k                               # SAI: thừa một lá ngang
        for s in range(n, c - 1, -1): ways[s] += ways[s - c]
        k += 1
    return ways[n]

for n_, _ in examples:
    print(f"n={n_}: sai -> {bug_cost_3k(n_)} | đúng -> {Solution().houseOfCards(n_)}")

n=16: sai -> 0 | đúng -> 2
n=2: sai -> 0 | đúng -> 1
n=4: sai -> 0 | đúng -> 0


**Vì sao sai:** chỉ có `k − 1` lá ngang **giữa** `k` tam giác. **Sửa:** `3k − 1`. Kiểm bằng ví dụ 2 (`n = 2` phải là 1).

### Lỗi 2 — Cho phép hai hàng cùng cỡ (knapsack **không giới hạn**)

In [17]:
def bug_unbounded(n):
    ways = [1] + [0] * n; k = 1
    while 3 * k - 1 <= n:
        c = 3 * k - 1
        for s in range(c, n + 1):               # SAI: duyệt tăng -> mỗi cỡ dùng nhiều lần
            ways[s] += ways[s - c]
        k += 1
    return ways[n]

for n_, _ in examples:
    print(f"n={n_}: sai -> {bug_unbounded(n_)} | đúng -> {Solution().houseOfCards(n_)}")

n=16: sai -> 6 | đúng -> 2
n=2: sai -> 1 | đúng -> 1
n=4: sai -> 1 | đúng -> 0


**Vì sao sai:** `n = 4 = 2 + 2` bị đếm (hai hàng 1 tam giác chồng nhau) — không hợp lệ vì tam giác hàng trên cần lá ngang ở dưới. **Sửa:** duyệt `s` **giảm dần**.

### Lỗi 3 — Đếm **thứ tự** các hàng (hoán vị)

In [18]:
def bug_ordered(n):
    @lru_cache(maxsize=None)
    def f(remain, used):
        if remain == 0: return 1
        total = 0
        for k in range(1, (remain + 1) // 3 + 1):
            if not used >> k & 1 and 3 * k - 1 <= remain:
                total += f(remain - (3 * k - 1), used | 1 << k)     # SAI: thứ tự hàng là cố định (giảm dần)
        return total
    return f(n, 0)

for n_ in [16, 15, 30]:
    print(f"n={n_}: sai -> {bug_ordered(n_)} | đúng -> {Solution().houseOfCards(n_)}")

n=16: sai -> 4 | đúng -> 2
n=15: sai -> 6 | đúng -> 1
n=30: sai -> 30 | đúng -> 5


**Vì sao sai:** với một tập cỡ hàng, chỉ có **một** cách xếp hợp lệ (cỡ lớn ở dưới). Đếm hoán vị nhân đáp án lên `r!`. **Sửa:** ép thứ tự (quay lui với `max_k` giảm) hoặc đếm tập (knapsack).

### Lỗi 4 — Ràng buộc không nghiêm ngặt: hàng trên `≤` hàng dưới

In [19]:
def bug_non_strict(n):
    @lru_cache(maxsize=None)
    def f(remain, m):
        if remain == 0: return 1
        if m == 0: return 0
        res = f(remain, m - 1)
        if 3 * m - 1 <= remain:
            res += f(remain - (3 * m - 1), m)        # SAI: lẽ ra m - 1 (hàng trên phải nhỏ hơn hẳn)
        return res
    return f(n, (n + 1) // 3)

for n_, _ in examples:
    print(f"n={n_}: sai -> {bug_non_strict(n_)} | đúng -> {Solution().houseOfCards(n_)}")

n=16: sai -> 6 | đúng -> 2
n=2: sai -> 1 | đúng -> 1
n=4: sai -> 1 | đúng -> 0


**Sửa:** hàng trên `k' ≤ k − 1` ⇒ chuyển sang `m − 1` sau khi dùng cỡ `m`. Lỗi này tương đương lỗi 2 dưới dạng đệ quy.

### Lỗi 5 — Quên rằng phải dùng **hết** `n` lá (đếm mọi nhà dùng `≤ n` lá)

In [20]:
def bug_at_most(n):
    ways = [1] + [0] * n; k = 1
    while 3 * k - 1 <= n:
        c = 3 * k - 1
        for s in range(n, c - 1, -1): ways[s] += ways[s - c]
        k += 1
    return sum(ways[1:])                         # SAI: đề yêu cầu dùng đúng n lá

for n_, _ in examples:
    print(f"n={n_}: sai -> {bug_at_most(n_)} | đúng -> {Solution().houseOfCards(n_)}")

n=16: sai -> 12 | đúng -> 2
n=2: sai -> 1 | đúng -> 1
n=4: sai -> 1 | đúng -> 0


**Sửa:** trả `ways[n]`. Đọc kỹ "using **all** n cards".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài hình học / mô tả dài ⇒ **vẽ** vài trường hợp nhỏ, đếm tay, rút công thức (`3k − 1`).
- "Dãy giảm nghiêm ngặt" ⇔ "tập con" ⇒ **knapsack 0/1** (duyệt tổng giảm dần).
- "Dãy không tăng" ⇔ "đa tập" ⇒ knapsack không giới hạn (duyệt tăng dần).
- Đổi biến tổng tuyến tính `Σ(a·k + b)` ⇒ phân hoạch với số phần cố định.
- Ràng buộc `n ≤ 500` nhỏ ⇒ kiểm được **mọi** `n` — nhưng số nhà tăng rất nhanh (`n = 500` có ~2.5·10⁷ nhà), nên quay lui chỉ dùng tới `n ≈ 200`.

## 15. Biến thể của bài

1. **Hàng trên được phép bằng hàng dưới** (nếu luật cho phép đặt chồng) ⇒ knapsack không giới hạn — chính là lỗi 2.
2. **Đếm nhà bài có đúng `r` hàng** — cách 4 cho sẵn `D[r][(n + r)/3]`.
3. **Nhà bài cao nhất** dùng đúng `n` lá (số hàng lớn nhất) — bài tập 5.
4. **LeetCode 518 — Coin Change II / 416 — Partition Equal Subset Sum:** knapsack đếm / khả thi.
5. **Phân hoạch thành phần phân biệt** của `n` (số Euler `q(n)`) — cùng DP với chi phí `k`.

In [21]:
def distinct_partition_count(n):
    """Biến thể 5: q(n) = số cách viết n thành tổng các số nguyên dương phân biệt (knapsack chi phí k)."""
    ways = [1] + [0] * n
    for k in range(1, n + 1):
        for s in range(n, k - 1, -1):
            ways[s] += ways[s - k]
    return ways[n]

print("q(n) với n = 1..15:", [distinct_partition_count(n_) for n_ in range(1, 16)])
print("(dãy OEIS A000009: 1, 1, 2, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 22, 27)")
assert [distinct_partition_count(n_) for n_ in range(1, 16)] == [1, 1, 2, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 22, 27]

q(n) với n = 1..15: [1, 1, 2, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 22, 27]
(dãy OEIS A000009: 1, 1, 2, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 22, 27)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | liệt kê dãy giảm | mũ theo `√n` | `O(R)` | kiểm thử, in nhà |
| 2. DFS ghi nhớ | `f(remain, m)` | `O(nK)` | `O(nK)` | suy ra từ quay lui |
| 3. Knapsack 0/1 | đếm tập | `O(nK)` | `O(n)` | **nộp bài** |
| 4. Phân hoạch phân biệt | đổi biến | `O(R n)` | `O(R n)` | đếm theo số hàng |
| 5. Sinh hàm | `Π(1 + x^{3k−1})` | `O(nK)` | `O(n)` | suy biến thể nhanh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().houseOfCards(17)`?

**Bài 2 (tính tay):** Liệt kê mọi nhà bài với `n = 27`.

**Bài 3 (điền TODO):** Hoàn thiện `houses_with_rows(n, r)` — số nhà bài dùng đúng `n` lá và có đúng `r` hàng.

**Bài 4 (tự viết lại):** Viết lại cách 3 trả về **toàn bộ bảng** `ways[0..n]` và dùng nó trả lời nhanh nhiều truy vấn `n`.

**Bài 5 (mini challenge):** Với `n` cho trước, tìm **số hàng lớn nhất** của một nhà bài dùng đúng `n` lá (hoặc `0` nếu không có nhà nào). Kiểm bằng quay lui với mọi `n ≤ 120`.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def houses_with_rows(n, r):
    # TODO: (n + r) % 3 == 0 thì D[r][(n + r) // 3], ngược lại 0
    pass

# Bài 4 — TODO
def all_house_counts(N):
    # TODO: knapsack một lần cho N, trả về list ways
    pass

# Bài 5 — TODO
def max_rows(n):
    # TODO: DP theo (tổng, số hàng) hoặc dùng houses_with_rows với r giảm dần
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Chi phí: `2, 5, 8, 11, 14, 17`.
- Mức 2: Một hàng: `17 = 3·6 − 1` ✓. Hai hàng: `14 + 3`, `11 + 6`, `8 + 9` — phần còn lại không phải chi phí hợp lệ (hoặc trùng cỡ). Ba hàng: nhỏ nhất `2 + 5 + 8 = 15`, kế tiếp `2 + 5 + 11 = 18 > 17`.
- Mức 3: Chỉ có nhà một hàng 6 tam giác ⇒ `1`.

**Bài 2**
- Mức 1: `n + r ≡ 0 (mod 3)` ⇒ `r ∈ {3, 6, …}`.
- Mức 2: `r = 3`: `Σk = 10` thành 3 phần phân biệt.
- Mức 3: In bằng `list_houses(27)`.

**Bài 3**
- Mức 1: Dùng bảng `D` của cách 4.
- Mức 2: `rmax` đủ lớn: `r(3r + 1)/2 ≤ n` (nhà nhỏ nhất `r` hàng dùng `Σ_{k=1..r}(3k−1)`).
- Mức 3: Tổng theo `r` phải bằng `houseOfCards(n)`.

**Bài 4**
- Mức 1: Trả `ways` thay vì `ways[n]`.
- Mức 2: Mọi `n ≤ N` dùng chung một bảng.
- Mức 3: So với `Solution` cho từng `n`.

**Bài 5**
- Mức 1: Duyệt `r` từ lớn xuống, lấy `r` đầu tiên có `houses_with_rows(n, r) > 0`.
- Mức 2: Hoặc DP `best[s]` = số hàng lớn nhất với tổng `s` (knapsack max thay vì đếm).
- Mức 3: Kiểm bằng `max(len(h) for h in list_houses(n))`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = Solution().houseOfCards(17)
print("Bài 1:", r1, list_houses(17)); assert r1 == house_brute(17)

# Bài 2
print("Bài 2:", list_houses(27)); assert len(list_houses(27)) == Solution().houseOfCards(27)

# Bài 3
def houses_with_rows_sol(n, r):
    if (n + r) % 3 or r <= 0:
        return 0
    S = (n + r) // 3
    D = distinct_partitions_table(S, r)
    return D[r][S]

for n_ in range(1, 121):
    hs = list_houses(n_)
    for r_ in range(1, 20):
        assert houses_with_rows_sol(n_, r_) == sum(1 for h in hs if len(h) == r_)
print("Bài 3 OK; n=100 theo số hàng:", {r_: houses_with_rows_sol(100, r_) for r_ in range(1, 9) if houses_with_rows_sol(100, r_)})

Bài 1: 1 [(6,)]
Bài 2: [(7, 2, 1), (6, 3, 1), (5, 4, 1), (5, 3, 2)]
Bài 3 OK; n=100 theo số hàng: {2: 16, 5: 192, 8: 1}


In [24]:
# Bài 4
def all_house_counts_sol(N):
    ways = [1] + [0] * N; k = 1
    while 3 * k - 1 <= N:
        c = 3 * k - 1
        for s in range(N, c - 1, -1): ways[s] += ways[s - c]
        k += 1
    return ways

table = all_house_counts_sol(500)
assert all(table[n_] == Solution().houseOfCards(n_) for n_ in range(1, 501))

# Bài 5
def max_rows_sol(n):
    NEG = -1
    best = [0] + [NEG] * n; k = 1
    while 3 * k - 1 <= n:
        c = 3 * k - 1
        for s in range(n, c - 1, -1):
            if best[s - c] != NEG:
                best[s] = max(best[s], best[s - c] + 1)
        k += 1
    return max(best[n], 0)

for n_ in range(1, 121):
    hs = list_houses(n_)
    assert max_rows_sol(n_) == (max(len(h) for h in hs) if hs else 0)
print("Bài 4, Bài 5 OK; ví dụ số hàng lớn nhất:", {n_: max_rows_sol(n_) for n_ in [2, 16, 30, 100, 500]})

Bài 4, Bài 5 OK; ví dụ số hàng lớn nhất: {2: 1, 16: 2, 30: 3, 100: 8, 500: 16}


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template knapsack 0/1 đếm số cách:**

```python
ways = [1] + [0] * n
for cost in costs:                       # ở đây cost = 3k - 1, k = 1, 2, ...
    for s in range(n, cost - 1, -1):     # GIẢM dần
        ways[s] += ways[s - cost]
return ways[n]
```

**Checklist:**
- [ ] Số lá một hàng `3k − 1` (kiểm bằng `n = 2 → 1`)?
- [ ] Hàng trên **ít hơn hẳn** ⇒ mỗi cỡ dùng tối đa một lần (duyệt giảm)?
- [ ] Không đếm hoán vị các hàng?
- [ ] Dùng **đúng hết** `n` lá (`ways[n]`)?
- [ ] Đã kiểm mọi `n ≤ 200` với quay lui (và đối chiếu chéo tới 500)?
- [ ] Chữ ký đúng: `def houseOfCards(self, n: int) -> int`? (đối chiếu đề gốc nếu có Premium)